In [ ]:
import sys, json, os, yaml
from pathlib import Path
import numpy as np
root = Path(os.getcwd())
if (root / '_framework').exists():
    sys.path.insert(0, str(root / '_framework'))
else:
    sys.path.insert(0, str(root.parent / '_framework'))
import common
import brax_common as bc
cfg = yaml.safe_load(open('deliverables/config.yaml', encoding='utf-8'))
print('env:', cfg.get('env_name', cfg.get('env_id')), '| success 阈值:', json.dumps(cfg.get('success'), ensure_ascii=False))


In [ ]:
from stable_baselines3 import PPO
model = PPO.load('deliverables/best_model.zip', device=cfg.get('device', 'auto'))
print('model loaded on', model.device)
result = common.evaluate(cfg, model, 'deliverables/best_vecnormalize.pkl', n_episodes=10)
print(json.dumps({k: result[k] for k in ['mean_return', 'survival', 'mean_speed', 'mean_goal_dist', 'mean_final_z', 'success', 'success_reason']}, ensure_ascii=False, indent=2))


In [ ]:
common.render_rollout(cfg, model, 'deliverables/best_vecnormalize.pkl', 'my_rollout.mp4')
from IPython.display import Video
Video('my_rollout.mp4', embed=True, width=640)


In [ ]:
new_cfg = dict(cfg)
new_cfg['success'] = dict(new_cfg.get('success', {}))
print('原阈值:', new_cfg['success'])
new_cfg['success'] = {k: v * 0.9 if isinstance(v, (int, float)) and k != 'min_survival' else v for k, v in new_cfg['success'].items()}
print('放宽到 90%:', new_cfg['success'])
ok, reason = bc.check_success(new_cfg, result['mean_return'], result['survival'], result['mean_speed'], result['mean_goal_dist'], result['mean_final_z'])
print('判定:', ok, '|', reason)
